# Abgabe 3 – Kapitel 6: Entscheidungsbäume (Programmieraufgaben)

Programmieraufgaben 7–8 aus Hands-On Machine Learning (Géron), Kapitel 6.

## Setup

In [1]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

## Aufgabe 7 — Entscheidungsbaum auf dem moons-Datensatz

Datensatz erzeugen und aufteilen.

In [2]:
X, y = make_moons(n_samples=10000, noise=0.4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training:", X_train.shape, "| Test:", X_test.shape)

Training: (8000, 2) | Test: (2000, 2)


Gittersuche mit Kreuzvalidierung, vor allem über max_leaf_nodes.

In [3]:
params = {
    "max_leaf_nodes": [2, 5, 10, 15, 20, 25, 30, 40, 50, 100],
    "min_samples_split": [2, 3, 4],
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42), params, cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Beste Parameter:", grid_search.best_params_)
print(f"Beste CV-Genauigkeit: {grid_search.best_score_:.4f}")

Beste Parameter: {'max_leaf_nodes': 20, 'min_samples_split': 2}
Beste CV-Genauigkeit: 0.8550


Bestes Modell (von GridSearchCV automatisch nachtrainiert) auf den Testdaten bewerten.

In [4]:
tree_clf = grid_search.best_estimator_
tree_acc = accuracy_score(y_test, tree_clf.predict(X_test))
print(f"Genauigkeit auf den Testdaten: {tree_acc:.4f}")

Genauigkeit auf den Testdaten: 0.8700


Die Genauigkeit liegt wie erwartet zwischen 85 % und 87 %.

## Aufgabe 8 — Einen Wald züchten

1.000 Teildatensätze mit je 100 Datenpunkten per ShuffleSplit erzeugen.

In [5]:
from sklearn.model_selection import ShuffleSplit

n_trees = 1000
n_instances = 100

mini_sets = []
rs = ShuffleSplit(n_splits=n_trees, test_size=len(X_train) - n_instances,
                  random_state=42)
for mini_train_index, mini_test_index in rs.split(X_train):
    mini_sets.append((X_train[mini_train_index], y_train[mini_train_index]))

print(f"{len(mini_sets)} Teildatensätze mit je {len(mini_sets[0][0])} Datenpunkten.")

1000 Teildatensätze mit je 100 Datenpunkten.


Auf jedem Teildatensatz einen Baum mit den besten Hyperparametern trainieren und einzeln auswerten.

In [6]:
from sklearn.base import clone

forest = [clone(grid_search.best_estimator_) for _ in range(n_trees)]

accuracy_scores = []
for tree, (X_mini, y_mini) in zip(forest, mini_sets):
    tree.fit(X_mini, y_mini)
    accuracy_scores.append(accuracy_score(y_test, tree.predict(X_test)))

print(f"Durchschnittliche Genauigkeit der Einzelbäume: {np.mean(accuracy_scores):.4f}")

Durchschnittliche Genauigkeit der Einzelbäume: 0.8012


Die auf nur 100 Punkten trainierten Einzelbäume erreichen im Schnitt nur etwa 80 %.

Für jeden Testpunkt die häufigste Vorhersage der 1.000 Bäume behalten (Mehrheitsvotum).

In [7]:
from scipy.stats import mode

Y_pred = np.empty([n_trees, len(X_test)], dtype=np.uint8)
for tree_index, tree in enumerate(forest):
    Y_pred[tree_index] = tree.predict(X_test)

y_pred_majority, n_votes = mode(Y_pred, axis=0, keepdims=False)
forest_acc = accuracy_score(y_test, y_pred_majority)
print(f"Genauigkeit des Mehrheitsvotums (Random Forest): {forest_acc:.4f}")

Genauigkeit des Mehrheitsvotums (Random Forest): 0.8720


In [8]:
print(f"Einzelner Baum (Aufgabe 7):      {tree_acc:.4f}")
print(f"Mittel der 1000 Einzelbäume:     {np.mean(accuracy_scores):.4f}")
print(f"Mehrheitsvotum (Random Forest):  {forest_acc:.4f}")
print(f"Verbesserung gegenüber dem ersten Baum: {(forest_acc - tree_acc)*100:+.2f} Prozentpunkte")

Einzelner Baum (Aufgabe 7):      0.8700
Mittel der 1000 Einzelbäume:     0.8012
Mehrheitsvotum (Random Forest):  0.8720
Verbesserung gegenüber dem ersten Baum: +0.20 Prozentpunkte


Das Mehrheitsvotum liegt etwas über dem einzelnen Baum. Damit wurde von Hand ein
Random-Forest-Klassifikator gebaut.